In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder.appName("AnalyseProcessed").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 12:20:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 12:20:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from pathlib import Path

# Resolve absolute path to data/raw from the notebook location
notebook_dir = Path().resolve()  # notebooks/
data_path = notebook_dir.parent / "data" / "raw"

In [4]:
print(f"Current dir: {notebook_dir}")
print(f"Data path: {data_path}")
print(f"Path exists: {data_path.exists()}")
print(f"JSON files found: {list(data_path.glob('*.json'))}")

Current dir: /Users/snehamungre/projects/crypto_market_analysis/notebooks
Data path: /Users/snehamungre/projects/crypto_market_analysis/data/raw
Path exists: True
JSON files found: [PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-03_09-48-58.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-06_14-03-20.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-02-27_10-24-30.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-02-26_08-43-05.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-05_14-02-47.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_data_raw_2026-03-08_12-47-59.json'), PosixPath('/Users/snehamungre/projects/crypto_market_analysis/data/raw/crypto_market_d

In [5]:
from pyspark.sql.types import *

schema = StructType(
    [
        StructField("ath", DoubleType(), True),
        StructField("ath_change_percentage", DoubleType(), True),
        StructField("ath_date", StringType(), True),
        StructField("atl", DoubleType(), True),
        StructField("atl_change_percentage", DoubleType(), True),
        StructField("atl_date", StringType(), True),
        StructField("circulating_supply", DoubleType(), True),
        StructField("current_price", DoubleType(), True),
        StructField("fully_diluted_valuation", LongType(), True),
        StructField("high_24h", DoubleType(), True),
        StructField("id", StringType(), True),
        StructField("image", StringType(), True),
        StructField("last_updated", StringType(), True),
        StructField("low_24h", DoubleType(), True),
        StructField("market_cap", LongType(), True),
        StructField("market_cap_change_24h", DoubleType(), True),
        StructField("market_cap_change_percentage_24h", DoubleType(), True),
        StructField("market_cap_rank", LongType(), True),
        StructField("max_supply", DoubleType(), True),
        StructField("name", StringType(), True),
        StructField("price_change_24h", DoubleType(), True),
        StructField("price_change_percentage_24h", DoubleType(), True),
        StructField(
            "roi",
            StructType(
                [
                    StructField("currency", StringType(), True),
                    StructField("percentage", DoubleType(), True),
                    StructField("times", DoubleType(), True),
                ]
            ),
            True,
        ),
        StructField(
            "sparkline_in_7d",
            StructType(
                [
                    StructField("price", ArrayType(DoubleType()), True),
                ]
            ),
            True,
        ),
        StructField("symbol", StringType(), True),
        StructField("total_supply", DoubleType(), True),
        StructField("total_volume", DoubleType(), True),
    ]
)

In [12]:
from pyspark.sql.functions import to_timestamp

df = (
    spark.read.option("multiLine", True)
    .option("header", True)
    .schema(schema)
    .json(str(data_path))
)

df = (
    df.withColumn("last_updated", to_timestamp("last_updated"))
    .withColumn("ath_date", to_timestamp("ath_date"))
    .withColumn("atl_date", to_timestamp("atl_date"))
    .drop("image", "symbol", "roi")
)

df.show(7)
df.count()

+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+-------------+-----------------------+--------+-----------+--------------------+--------+-------------+---------------------+--------------------------------+---------------+----------+--------+--------------------+---------------------------+--------------------+--------------------+---------------+
|     ath|ath_change_percentage|            ath_date|       atl|atl_change_percentage|            atl_date|  circulating_supply|current_price|fully_diluted_valuation|high_24h|         id|        last_updated| low_24h|   market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|    name|    price_change_24h|price_change_percentage_24h|     sparkline_in_7d|        total_supply|   total_volume|
+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+------------

1000

In [13]:
from pyspark.sql import functions as F

# Access the column and field using F.col
price_array = F.col("sparkline_in_7d.price")

# finds mean of the 7 day prices obtained
df = df.withColumn(
    "7d_avg",
    F.aggregate(price_array, F.lit(0.0), lambda acc, x: acc + x) / F.size(price_array),
)

# finds high of the 7 day prices obtained
df = df.withColumn("7d_max", F.array_max(price_array))

# finds low of the 7 day prices obtained
df = df.withColumn("7d_low", F.array_min(price_array))


df.select("7d_low", "7d_avg", "7d_max").show(5, truncate=False)

df = df.drop("sparkline_in_7d")

+------------------+------------------+------------------+
|7d_low            |7d_avg            |7d_max            |
+------------------+------------------+------------------+
|63176.9334493685  |68237.85767813282 |73669.77886917477 |
|1841.2881843632538|2003.6546057636438|2179.3034097539403|
|0.9996604391196411|1.0000409466281306|1.0004365229099252|
|591.2802216239186 |630.7528461470908 |664.0544992300274 |
|1.2779557957398355|1.3796730808820719|1.4591620419556068|
+------------------+------------------+------------------+
only showing top 5 rows


26/03/09 12:06:50 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 401746 ms exceeds timeout 120000 ms
26/03/09 12:06:51 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/09 12:06:53 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
# Ranking by current price
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("id").orderBy(desc("last_updated"))

most_updated_data = (
    df.withColumn("rn", row_number().over(window_spec)).filter("rn = 1").drop("rn")
)

In [ ]:
ranking_df = most_updated_data[
    "name", "current_price", "market_cap", "last_updated", "total_volume"
]

ranking_df.show(4)

+--------+-------------+----------+--------------------+------------+
|    name|current_price|market_cap|        last_updated|total_volume|
+--------+-------------+----------+--------------------+------------+
|    A7A5|   0.01266454| 496374281|2026-03-06 14:03:...|     2533.95|
|    Aave|       116.23|1764521280|2026-03-06 14:03:...|3.50139267E8|
|Algorand|     0.086592| 769498898|2026-03-06 14:03:...| 2.6021627E7|
|   Aptos|          1.0| 779980657|2026-03-06 14:03:...| 8.4650401E7|
+--------+-------------+----------+--------------------+------------+
only showing top 4 rows


## Ranking current price

In [ ]:
ranking_df.sort(desc("current_price")).show(10)

+------------+-------------+-------------+--------------------+---------------+
|        name|current_price|   market_cap|        last_updated|   total_volume|
+------------+-------------+-------------+--------------------+---------------+
|     Bitcoin|      70595.0|1411821473660|2026-03-06 14:03:...|4.9953784921E10|
|    PAX Gold|      5098.49|   2526367317|2026-03-06 14:03:...|   3.65138202E8|
| Tether Gold|      5058.89|   2855506165|2026-03-06 14:03:...|   7.42903717E8|
|    Ethereum|      2061.64| 248837730230|2026-03-06 14:03:...|2.0296994252E10|
|         BNB|       640.71|  87385587643|2026-03-06 14:03:...|  1.028567981E9|
|Bitcoin Cash|       456.78|   9133026670|2026-03-06 14:03:...|   2.51567996E8|
|      Monero|       357.12|   6588795379|2026-03-06 14:03:...|   1.17437508E8|
|       Zcash|       225.65|   3741392395|2026-03-06 14:03:...|   2.98168699E8|
|   Bittensor|       184.16|   1764617699|2026-03-06 14:03:...|   1.03349377E8|
|        Aave|       116.23|   176452128

## Ranking Market Cap

In [ ]:
ranking_df.sort(desc("market_cap")).show(10)

+------------+-------------+-------------+--------------------+---------------+
|        name|current_price|   market_cap|        last_updated|   total_volume|
+------------+-------------+-------------+--------------------+---------------+
|     Bitcoin|      70595.0|1411821473660|2026-03-06 14:03:...|4.9953784921E10|
|    Ethereum|      2061.64| 248837730230|2026-03-06 14:03:...|2.0296994252E10|
|      Tether|          1.0| 184040253579|2026-03-06 14:03:...|7.6978208281E10|
|         BNB|       640.71|  87385587643|2026-03-06 14:03:...|  1.028567981E9|
|         XRP|          1.4|  85507969928|2026-03-06 14:03:...|  2.410070684E9|
|        USDC|     0.999975|  77159263074|2026-03-06 14:03:...|1.0996329558E10|
|      Solana|        87.47|  49853763964|2026-03-06 14:03:...|  4.348826287E9|
|        TRON|     0.286461|  27137648168|2026-03-06 14:03:...|   5.46974368E8|
|Figure Heloc|        1.035|  16144541020|2026-03-06 12:25:...|    2.3413865E8|
|    Dogecoin|      0.09322|  1428859841